In [1]:
# ---------------------------------------------------------
# Step 1: Faithfulness evaluation setup
# ---------------------------------------------------------

from pathlib import Path
import json
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
)

EVALUATION_DIR = (
    RESULTS_DIR
    / "evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [10]:
# ---------------------------------------------------------
# Input paths
# ---------------------------------------------------------

CLINICAL_NOTES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

STUDY_PATIENT_IDS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "patients"
    / "study_patient_ids.json"
)

In [4]:

# ---------------------------------------------------------
# Faithfulness output paths
# ---------------------------------------------------------

FAITHFULNESS_CLAIMS_PATH = (
    EVALUATION_DIR
    / "faithfulness_atomic_claims.json"
)

FAITHFULNESS_RESULTS_PATH = (
    EVALUATION_DIR
    / "faithfulness_results.json"
)

In [7]:
from openai import OpenAI

client = OpenAI()

In [8]:
# ---------------------------------------------------------
# Evaluator model
# ---------------------------------------------------------

EVALUATOR_MODEL = "gpt-5.4-mini"


print("Project root:", PROJECT_ROOT)
print("Evaluation directory:", EVALUATION_DIR)
print("Evaluator model:", EVALUATOR_MODEL)

Project root: /Users/pallavi_chandanshive/projects/clinical-summarization-eval
Evaluation directory: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation
Evaluator model: gpt-5.4-mini


In [11]:
# ---------------------------------------------------------
# Step 2: Load frozen cohort and four workflow summaries
# ---------------------------------------------------------

# Load the exact frozen 50-patient cohort.
with open(STUDY_PATIENT_IDS_PATH, "r") as f:
    study_patient_ids = json.load(f)

study_patient_id_set = set(study_patient_ids)

print("Study patients:", len(study_patient_ids))

Study patients: 50


In [12]:
# ---------------------------------------------------------
# Final workflow result paths
# ---------------------------------------------------------

workflow_paths = {
    "direct": (
        RESULTS_DIR
        / "direct"
        / "direct_summaries.json"
    ),
    "hierarchical": (
        RESULTS_DIR
        / "hierarchical"
        / "hierarchical_summaries.json"
    ),
    "rag": (
        RESULTS_DIR
        / "rag"
        / "final_rag_summaries.json"
    ),
    "rag_verification": (
        RESULTS_DIR
        / "rag_verification"
        / "final_verified_summaries.json"
    ),
}

In [13]:

# Each workflow stores its final summary under a different field.
summary_fields = {
    "direct": "summary",
    "hierarchical": "final_summary",
    "rag": "summary",
    "rag_verification": "final_summary",
}


In [14]:
# ---------------------------------------------------------
# Load summaries
# ---------------------------------------------------------

workflow_summaries = {}

for workflow, path in workflow_paths.items():

    with open(path, "r") as f:
        results = json.load(f)

    summary_field = summary_fields[workflow]

    workflow_summaries[workflow] = {
        person_id: data[summary_field]
        for person_id, data in results.items()
    }


In [15]:
# ---------------------------------------------------------
# Validate exact cohort for every workflow
# ---------------------------------------------------------

for workflow, summaries in workflow_summaries.items():

    workflow_ids = set(summaries.keys())

    print(
        workflow,
        "| summaries:",
        len(summaries),
        "| IDs match:",
        workflow_ids == study_patient_id_set
    )

direct | summaries: 50 | IDs match: True
hierarchical | summaries: 50 | IDs match: True
rag | summaries: 50 | IDs match: True
rag_verification | summaries: 50 | IDs match: True


In [16]:
# ---------------------------------------------------------
# Step 3: Reconstruct complete original patient records
# ---------------------------------------------------------

# Load original clinical notes.
notes = pd.read_csv(CLINICAL_NOTES_PATH)


# ---------------------------------------------------------
# Apply the exact frozen preprocessing used in the study
# ---------------------------------------------------------

# Remove notes whose entire cleaned text is "#NAME?".
notes_clean = notes[
    notes["clean_note_text"].astype(str).str.strip() != "#NAME?"
].copy()


# Sort chronologically and remove duplicate notes.
# If the same note text occurs more than once for a patient,
# retain only its earliest occurrence.
notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)


# Keep only the frozen 50-patient study cohort.
notes_dedup = notes_dedup[
    notes_dedup["person_id"].isin(study_patient_id_set)
].copy()


# ---------------------------------------------------------
# Build one complete source document per patient
# ---------------------------------------------------------

patient_source_documents = (
    notes_dedup
    .groupby("person_id", sort=False)["clean_note_text"]
    .apply(
        lambda texts: "\n\n".join(
            texts.astype(str)
        )
    )
    .to_dict()
)


# ---------------------------------------------------------
# Validate
# ---------------------------------------------------------

source_patient_ids = set(patient_source_documents.keys())

print("Source documents:", len(patient_source_documents))
print("Study patients:", len(study_patient_ids))
print(
    "IDs match:",
    source_patient_ids == study_patient_id_set
)

print(
    "Missing IDs:",
    study_patient_id_set - source_patient_ids
)

print(
    "Extra IDs:",
    source_patient_ids - study_patient_id_set
)

Source documents: 50
Study patients: 50
IDs match: True
Missing IDs: set()
Extra IDs: set()


In [17]:
# ---------------------------------------------------------
# Step 4A: Atomic claim extraction prompt
# ---------------------------------------------------------

FAITHFULNESS_CLAIM_SYSTEM_PROMPT = """
You are extracting atomic factual claims from a generated clinical summary
for a faithfulness evaluation.

Your task is ONLY to decompose the summary into independently verifiable
clinical claims.

An atomic claim should contain one factual proposition that can be checked
against the patient's original clinical record.

IMPORTANT RULES:

1. Preserve the exact clinical meaning of the generated summary.
2. Do NOT add, infer, correct, or reinterpret information.
3. Do NOT use outside medical knowledge.
4. Preserve important relationships between entities.

   For example, if a sentence states that medication A was given for
   condition B, the extracted claim must preserve that medication-condition
   relationship correctly.

5. When a modifier, indication, dose, timing, anatomical site, or other
   detail clearly belongs to a specific entity, keep it attached to that
   entity.

6. Split compound statements only when doing so does not change their
   meaning or create incorrect relationships.

7. Do not make a claim more specific than the original summary.
8. Do not make a claim more general than the original summary.
9. Include factual clinical statements such as diagnoses, symptoms,
   investigations, treatments, procedures, medication changes, clinical
   progression, and outcomes.
10. Exclude purely structural headings or non-factual filler.

Return ONLY valid JSON in this format:

{
  "claims": [
    {
      "claim_id": 1,
      "claim": "..."
    },
    {
      "claim_id": 2,
      "claim": "..."
    }
  ]
}
""".strip()

In [18]:
# ---------------------------------------------------------
# Step 4B: Select one stress-test summary
# ---------------------------------------------------------

PILOT_PERSON_ID = "05192757-942f-460d-b4ff-004ec39cc5ee"
PILOT_WORKFLOW = "rag"

pilot_summary = workflow_summaries[
    PILOT_WORKFLOW
][PILOT_PERSON_ID]

print("Patient:", PILOT_PERSON_ID)
print("Workflow:", PILOT_WORKFLOW)
print()
print(pilot_summary)

Patient: 05192757-942f-460d-b4ff-004ec39cc5ee
Workflow: rag

- **21/12/2025 – Preoperative assessment:** Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis, associated with stiffness and limited mobility. Medical history included hypertension, severe left hip OA, and cataract surgery in 2018. BP was 145/85 mmHg and Hb 11.3 g/dL, consistent with mild anaemia; she was taking ferrous sulfate 200 mg daily, paracetamol as needed, and calcium/vitamin D supplements. ECG showed normal sinus rhythm without ischaemic changes. No allergies were reported. She was deemed suitable for spinal anaesthesia, consented for elective total left hip replacement, and cleared for admission on 02/01/2026.

- **02/01/2026 – Admission and surgery:** Preoperative checks confirmed identity, fasting status, correct left hip site, medication reconciliation, and suitability to proceed. Mild anaemia had been optimised following ferrous sulfate, and elevated BP was c

In [19]:
# ---------------------------------------------------------
# Step 4C: Pilot atomic claim extraction
# ---------------------------------------------------------

pilot_user_prompt = f"""
GENERATED CLINICAL SUMMARY:

{pilot_summary}
""".strip()


response = client.chat.completions.create(
    model=EVALUATOR_MODEL,
    messages=[
        {
            "role": "system",
            "content": FAITHFULNESS_CLAIM_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": pilot_user_prompt,
        },
    ],
    response_format={"type": "json_object"},
)


# Parse the returned JSON.
pilot_claim_result = json.loads(
    response.choices[0].message.content
)

pilot_claims = pilot_claim_result["claims"]


print("Number of claims:", len(pilot_claims))
print()

for claim in pilot_claims:
    print(
        claim["claim_id"],
        "-",
        claim["claim"]
    )

Number of claims: 84

1 - On 21/12/2025, Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis.
2 - Her left hip pain was associated with stiffness and limited mobility.
3 - Her medical history included hypertension.
4 - Her medical history included severe left hip osteoarthritis.
5 - Her medical history included cataract surgery in 2018.
6 - At preoperative assessment, her blood pressure was 145/85 mmHg.
7 - At preoperative assessment, her haemoglobin was 11.3 g/dL.
8 - Her haemoglobin of 11.3 g/dL was consistent with mild anaemia.
9 - She was taking ferrous sulfate 200 mg daily.
10 - She was taking paracetamol as needed.
11 - She was taking calcium and vitamin D supplements.
12 - Her ECG showed normal sinus rhythm without ischaemic changes.
13 - No allergies were reported.
14 - She was deemed suitable for spinal anaesthesia.
15 - She consented to elective total left hip replacement.
16 - She was cleared for admission on 02/01/2026.
17

In [20]:
# ---------------------------------------------------------
# Step 5: Extract atomic claims from all final summaries
# ---------------------------------------------------------

# Load existing progress if this cell is being resumed.
if FAITHFULNESS_CLAIMS_PATH.exists():

    with open(FAITHFULNESS_CLAIMS_PATH, "r") as f:
        faithfulness_atomic_claims = json.load(f)

    print("Loaded existing claim extraction progress.")

else:
    faithfulness_atomic_claims = {}


# ---------------------------------------------------------
# Extract claims for each patient × workflow
# ---------------------------------------------------------

for workflow, summaries in workflow_summaries.items():

    # Create workflow container if it does not already exist.
    if workflow not in faithfulness_atomic_claims:
        faithfulness_atomic_claims[workflow] = {}

    for person_id in study_patient_ids:

        # Skip summaries already processed.
        if person_id in faithfulness_atomic_claims[workflow]:
            continue

        summary = summaries[person_id]

        user_prompt = f"""
GENERATED CLINICAL SUMMARY:

{summary}
""".strip()

        response = client.chat.completions.create(
            model=EVALUATOR_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": FAITHFULNESS_CLAIM_SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content": user_prompt,
                },
            ],
            response_format={"type": "json_object"},
        )

        result = json.loads(
            response.choices[0].message.content
        )

        claims = result["claims"]

        faithfulness_atomic_claims[workflow][person_id] = claims

        # Save after every completed summary so progress
        # survives notebook interruption.
        with open(FAITHFULNESS_CLAIMS_PATH, "w") as f:
            json.dump(
                faithfulness_atomic_claims,
                f,
                indent=2
            )

        print(
            workflow,
            person_id,
            "| claims:",
            len(claims)
        )


print("\nClaim extraction complete.")

direct 028998ee-babc-4096-9b28-001bc2f9a84e | claims: 53
direct 04df53ea-55c1-48d9-84a1-1f15c133b29b | claims: 59
direct 05192757-942f-460d-b4ff-004ec39cc5ee | claims: 69
direct 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf | claims: 86
direct 0f438665-d430-4adb-8acc-c3beed9e4942 | claims: 61
direct 136c7916-4f9b-4e5c-bf01-77e9d2c681a2 | claims: 52
direct 137b8481-4f1d-4b7f-babd-20f7117023ad | claims: 49
direct 1705dd0f-011a-492c-b006-b27e03f2f4ed | claims: 60
direct 1dbe23dc-0d1e-431b-81eb-497282b46a14 | claims: 67
direct 28570119-9cdc-4120-98c0-4edb76cf36a3 | claims: 53
direct 29ea304f-821d-474e-81a1-394ca3945e02 | claims: 67
direct 31f9612b-5a6b-48ea-887b-895772a83b99 | claims: 57
direct 359014a1-10e6-4bd8-9ba7-513d021c971e | claims: 51
direct 37b5ce4d-dcfd-4bb7-bee4-d597eb114703 | claims: 62
direct 420df33b-0072-4124-b1c6-0589daef3677 | claims: 104
direct 42149ae1-6a3c-471e-a002-cb7263e8bb8c | claims: 90
direct 51f15281-8840-4fd0-92de-89188ab8d736 | claims: 49
direct 58b8aad6-7327-4450-956f

In [21]:
# ---------------------------------------------------------
# Step 6: Validate atomic claim extraction
# ---------------------------------------------------------

# Reload from disk so validation checks the saved artifact.
with open(FAITHFULNESS_CLAIMS_PATH, "r") as f:
    faithfulness_atomic_claims = json.load(f)


total_claims = 0
claim_counts = []

for workflow in workflow_summaries:

    workflow_claims = faithfulness_atomic_claims.get(
        workflow,
        {}
    )

    workflow_ids = set(workflow_claims.keys())

    print(
        workflow,
        "| patients:",
        len(workflow_claims),
        "| IDs match:",
        workflow_ids == study_patient_id_set
    )

    for person_id, claims in workflow_claims.items():

        total_claims += len(claims)

        claim_counts.append({
            "workflow": workflow,
            "person_id": person_id,
            "claim_count": len(claims),
        })


claim_counts_df = pd.DataFrame(claim_counts)


print("\nTotal claims:", total_claims)

print("\nClaims per workflow:")
print(
    claim_counts_df
    .groupby("workflow")["claim_count"]
    .agg(["count", "sum", "mean", "min", "max"])
    .round(2)
)


# Check basic claim structure.
invalid_claims = []

for workflow, patients in faithfulness_atomic_claims.items():

    for person_id, claims in patients.items():

        for claim in claims:

            if (
                "claim_id" not in claim
                or "claim" not in claim
                or not str(claim["claim"]).strip()
            ):
                invalid_claims.append(
                    (workflow, person_id, claim)
                )


print("\nInvalid claims:", len(invalid_claims))

direct | patients: 50 | IDs match: True
hierarchical | patients: 50 | IDs match: True
rag | patients: 50 | IDs match: True
rag_verification | patients: 50 | IDs match: True

Total claims: 10970

Claims per workflow:
                  count   sum   mean  min  max
workflow                                      
direct               50  3000  60.00   34  104
hierarchical         50  2905  58.10   24  110
rag                  50  2658  53.16   34   81
rag_verification     50  2407  48.14   23   76

Invalid claims: 56


In [22]:
# ---------------------------------------------------------
# Step 6A: Inspect invalid claims
# ---------------------------------------------------------

print("Invalid claims:", len(invalid_claims))
print()

for workflow, person_id, claim in invalid_claims:
    print("Workflow:", workflow)
    print("Patient:", person_id)
    print("Claim:", claim)
    print("-" * 80)

Invalid claims: 56

Workflow: direct
Patient: 028998ee-babc-4096-9b28-001bc2f9a84e
Claim: {'claim': 'Macrogol was initially documented as one sachet three times daily.'}
--------------------------------------------------------------------------------
Workflow: direct
Patient: 028998ee-babc-4096-9b28-001bc2f9a84e
Claim: {'claim': 'He was encouraged to maintain oral hydration.'}
--------------------------------------------------------------------------------
Workflow: direct
Patient: 028998ee-babc-4096-9b28-001bc2f9a84e
Claim: {'claim': 'He was monitored for bowel movements and symptoms.'}
--------------------------------------------------------------------------------
Workflow: direct
Patient: 04df53ea-55c1-48d9-84a1-1f15c133b29b
Claim: {'claim': 'He reported no loss of consciousness, vomiting, visual symptoms, seizures, or focal neurological symptoms.'}
--------------------------------------------------------------------------------
Workflow: direct
Patient: 61699d6d-904a-4ece-9026-cd6

In [23]:
# ---------------------------------------------------------
# Step 6B: Normalize claim IDs deterministically
# ---------------------------------------------------------

for workflow, patients in faithfulness_atomic_claims.items():

    for person_id, claims in patients.items():

        # Assign sequential IDs according to the order
        # returned by the claim extractor.
        for claim_id, claim in enumerate(claims, start=1):
            claim["claim_id"] = claim_id


# Save the normalized claims.
with open(FAITHFULNESS_CLAIMS_PATH, "w") as f:
    json.dump(
        faithfulness_atomic_claims,
        f,
        indent=2
    )


print("Claim IDs normalized and saved.")

Claim IDs normalized and saved.


In [24]:
# ---------------------------------------------------------
# Step 6C: Revalidate normalized claims
# ---------------------------------------------------------

invalid_claims = []

for workflow, patients in faithfulness_atomic_claims.items():

    for person_id, claims in patients.items():

        for claim in claims:

            if (
                "claim_id" not in claim
                or "claim" not in claim
                or not str(claim["claim"]).strip()
            ):
                invalid_claims.append(
                    (workflow, person_id, claim)
                )


print("Invalid claims:", len(invalid_claims))

Invalid claims: 0


In [28]:
# ---------------------------------------------------------
# Step 7D: Locate notebooks containing section chunking
# ---------------------------------------------------------

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

for path in sorted(NOTEBOOKS_DIR.rglob("*.ipynb")):
    text = path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).lower()

    if (
        "section_chunks" in text
        or "section-aware" in text
        or "section_chunk" in text
    ):
        print(path.relative_to(PROJECT_ROOT))

notebooks/evaluation/01_tfidf_evaluation.ipynb
notebooks/experiments/03_workflow_2_rag_experiments.ipynb
notebooks/rag/01_configuration_summaries.ipynb
notebooks/rag/02_configuration_evaluation.ipynb
notebooks/rag/03_rag_implementation.ipynb
notebooks/rag_verification/rag_verification.ipynb


In [29]:
# ---------------------------------------------------------
# Step 7E: Inspect chunking code from Workflow 4
# ---------------------------------------------------------

import nbformat

VERIFICATION_NOTEBOOK_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "rag_verification"
    / "rag_verification.ipynb"
)

verification_nb = nbformat.read(
    VERIFICATION_NOTEBOOK_PATH,
    as_version=4
)

for i, cell in enumerate(verification_nb.cells):

    if cell.cell_type != "code":
        continue

    source = cell.source.lower()

    if (
        "chunk" in source
        or "bge" in source
        or "embedding" in source
    ):
        print(f"\n{'=' * 80}")
        print("CELL:", i)
        print("=" * 80)
        print(cell.source)


CELL: 13
bge_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

CELL: 14
def split_note_into_sections(note_text):
    """
    Split a clinical note using explicit section headings already
    present in the note.

    Returns:
        list of dictionaries:
        [
            {
                "section_name": "...",
                "chunk_text": "..."
            },
            ...
        ]
    """

    text = str(note_text).strip()

    if not text:
        return []

    section_names = [
        "Chief Complaint",
        "Presenting Complaint",
        "History of Present Illness",
        "HPI",
        "Past Medical History",
        "PMH",
        "Past Surgical History",
        "PSH",
        "Medications",
        "Current Medications",
        "Allergies",
        "Family History",
        "Social History",
        "Review of Systems",
        "ROS",
        "Physical Examination",
        "Physical Exam",
        "Examination",
        "Vital Signs",
        "V

In [30]:
import re
import numpy as np

from sentence_transformers import SentenceTransformer

In [31]:
# ---------------------------------------------------------
# Step 7F: Rebuild section-aware chunks from frozen notes
# ---------------------------------------------------------

def split_note_into_sections(note_text):
    """
    Split a clinical note using explicit recognized section headings.

    If no recognized headings exist, retain the entire note as one
    unsectioned chunk.
    """

    text = str(note_text).strip()

    if not text:
        return []

    section_names = [
        "Chief Complaint",
        "Presenting Complaint",
        "History of Present Illness",
        "HPI",
        "Past Medical History",
        "PMH",
        "Past Surgical History",
        "PSH",
        "Medications",
        "Current Medications",
        "Allergies",
        "Family History",
        "Social History",
        "Review of Systems",
        "ROS",
        "Physical Examination",
        "Physical Exam",
        "Examination",
        "Vital Signs",
        "Vitals",
        "Investigations",
        "Laboratory Results",
        "Labs",
        "Imaging",
        "Assessment",
        "Impression",
        "Diagnosis",
        "Diagnoses",
        "Plan",
        "Assessment and Plan",
        "Treatment",
        "Hospital Course",
        "Clinical Course",
        "Discharge Plan",
        "Follow Up",
        "Follow-Up",
    ]

    heading_pattern = "|".join(
        re.escape(name)
        for name in sorted(
            section_names,
            key=len,
            reverse=True
        )
    )

    pattern = re.compile(
        rf"(?im)^[ \t]*(?P<heading>{heading_pattern})"
        rf"[ \t]*(?::|-)?[ \t]*$"
    )

    matches = list(pattern.finditer(text))

    # No recognized headings -> keep complete note.
    if not matches:
        return [{
            "section_name": "Unsectioned",
            "chunk_text": text,
        }]

    sections = []

    # Preserve any text before the first heading.
    prefix = text[:matches[0].start()].strip()

    if prefix:
        sections.append({
            "section_name": "Preamble",
            "chunk_text": prefix,
        })

    # Extract each recognized section.
    for i, match in enumerate(matches):

        section_name = match.group("heading").strip()
        content_start = match.end()

        if i + 1 < len(matches):
            content_end = matches[i + 1].start()
        else:
            content_end = len(text)

        content = text[
            content_start:content_end
        ].strip()

        if content:
            sections.append({
                "section_name": section_name,
                "chunk_text": f"{section_name}\n{content}",
            })

    return sections


# ---------------------------------------------------------
# Apply the chunker to the frozen deduplicated source notes
# ---------------------------------------------------------

section_records = []

for _, row in notes_dedup.iterrows():

    sections = split_note_into_sections(
        row["clean_note_text"]
    )

    for section in sections:

        section_records.append({
            "person_id": row["person_id"],
            "creation_timestamp": row["creation_timestamp"],
            "section_name": section["section_name"],
            "chunk_text": section["chunk_text"],
        })


section_chunks = pd.DataFrame(section_records)

section_chunks = (
    section_chunks
    .sort_values([
        "person_id",
        "creation_timestamp",
    ])
    .reset_index(drop=True)
)

section_chunks["chunk_id"] = range(
    len(section_chunks)
)


print("Total section chunks:", len(section_chunks))
print(
    "Patients:",
    section_chunks["person_id"].nunique()
)

Total section chunks: 2771
Patients: 50


In [32]:
# ---------------------------------------------------------
# Step 8: Embed complete source evidence with BGE
# ---------------------------------------------------------

bge_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

section_chunk_embeddings = bge_model.encode(
    section_chunks["chunk_text"].astype(str).tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Chunks:", len(section_chunks))
print(
    "Embedding shape:",
    section_chunk_embeddings.shape
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

Chunks: 2771
Embedding shape: (2771, 768)


In [33]:
# ---------------------------------------------------------
# Step 9: Define BGE Top-10 evidence retrieval
# ---------------------------------------------------------

FAITHFULNESS_TOP_K = 10


def retrieve_faithfulness_evidence(
    claim_text,
    patient_evidence,
    patient_embeddings,
    top_k=FAITHFULNESS_TOP_K
):
    """
    Retrieve the most semantically relevant original-record chunks
    for one generated-summary atomic claim.

    BGE is used only for evidence retrieval.
    It does NOT determine whether the claim is supported.
    """

    # Embed the generated-summary claim.
    claim_embedding = bge_model.encode(
        [claim_text],
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]

    # Cosine similarity because embeddings are normalized.
    similarities = (
        patient_embeddings
        @ claim_embedding
    )

    # Retrieve the Top-K most similar source chunks.
    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    retrieved = []

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):
        row = patient_evidence.iloc[int(idx)]

        retrieved.append({
            "rank": int(rank),
            "chunk_id": int(row["chunk_id"]),
            "creation_timestamp": str(
                row["creation_timestamp"]
            ),
            "section_name": str(
                row["section_name"]
            ),
            "text": str(
                row["chunk_text"]
            ),
            "similarity": float(
                similarities[int(idx)]
            ),
        })

    return retrieved

In [34]:
# ---------------------------------------------------------
# Step 10: Sanity-check Faithfulness evidence retrieval
# ---------------------------------------------------------

TEST_PERSON_ID = "05192757-942f-460d-b4ff-004ec39cc5ee"
TEST_WORKFLOW = "rag"

# Get this patient's original-record evidence.
patient_mask = (
    section_chunks["person_id"].astype(str)
    == str(TEST_PERSON_ID)
)

patient_evidence = (
    section_chunks[patient_mask]
    .copy()
    .reset_index(drop=True)
)

patient_embeddings = (
    section_chunk_embeddings[
        patient_mask.to_numpy()
    ]
)

# Get the already-saved Faithfulness atomic claims.
test_claims = (
    faithfulness_atomic_claims[
        TEST_WORKFLOW
    ][TEST_PERSON_ID]
)

print("Patient evidence chunks:", len(patient_evidence))
print("Atomic claims:", len(test_claims))


# Inspect a few claims spread across the summary.
test_indices = [
    0,
    len(test_claims) // 2,
    len(test_claims) - 1,
]

for idx in test_indices:

    claim = test_claims[idx]

    retrieved = retrieve_faithfulness_evidence(
        claim_text=claim["claim"],
        patient_evidence=patient_evidence,
        patient_embeddings=patient_embeddings,
    )

    print("\n" + "=" * 80)
    print(
        f"CLAIM {claim['claim_id']}: "
        f"{claim['claim']}"
    )

    for item in retrieved[:3]:
        print(
            f"\nRank {item['rank']} | "
            f"similarity={item['similarity']:.3f} | "
            f"{item['section_name']}"
        )
        print(item["text"][:700])

Patient evidence chunks: 47
Atomic claims: 55

CLAIM 1: On 21/12/2025, Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis.

Rank 1 | similarity=0.706 | Unsectioned
- Patient Name: Hope Chinwo
- Date of Birth: 23/03/1948 (Age: 75)
- NHS Number: 28704478
- Medical Record Number: 712349860
- Procedure: Total left hip replacement for end-stage OA
- Admission Type: Elective
- Admitting Consultant: Dr. Jennifer Jemma Hussein
- Ward: Orthopaedic Elective Ward 3A
- Bed Location: A07

Pre-operative checks completed by Nurse Deborah Eva Dickinson on 21/12/25 at 12:45:

- Patient identity confirmed using NHS number and wristband.
- Procedure and surgical site confirmed: Total left hip replacement.
- Allergies: None reported.
- Medical history reviewed: HTN, severe OA (left hip), cataract surgery (2018).
- Current medications reviewed: Paracetamol 500mg (as needed

Rank 2 | similarity=0.680 | Presenting Complaint
Presenting Complaint
Severe left

In [35]:
# ---------------------------------------------------------
# Step 11: Define Faithfulness support-judgment prompt
# ---------------------------------------------------------

FAITHFULNESS_JUDGE_SYSTEM_PROMPT = """
You are evaluating the faithfulness of a factual claim from a generated
clinical summary against evidence retrieved from the patient's original
clinical record.

Evaluate ONLY whether the claim is supported by the provided evidence.

IMPORTANT RULES:

1. Use only the provided clinical-record evidence.
2. Do NOT use outside medical knowledge.
3. Do NOT assume that a plausible statement is true.
4. Reasonable paraphrases and synonymous clinical wording count as support.
5. The evidence does not need to contain the exact wording of the claim.
6. Evaluate all clinically meaningful details in the claim, including
   diagnosis, medication, dose, timing, indication, anatomical site,
   procedure, investigation, progression, and outcome when present.
7. Absence of evidence is not evidence of support.
8. Do NOT use BGE similarity scores as evidence of correctness.
   Similarity was used only to retrieve candidate passages.

Assign exactly one label:

SUPPORTED:
The complete factual meaning of the claim is supported by the evidence.

PARTIALLY_SUPPORTED:
The central claim is supported, but one or more clinically meaningful
details are not supported by the evidence, OR only part of the factual
claim is supported.

UNSUPPORTED:
The claim is not supported by the provided evidence, or the evidence
contradicts the claim.

Return ONLY valid JSON:

{
  "label": "SUPPORTED | PARTIALLY_SUPPORTED | UNSUPPORTED",
  "reason": "Brief explanation based only on the provided evidence.",
  "supporting_evidence_ranks": [1, 2]
}
""".strip()


def judge_faithfulness_claim(
    claim_text,
    retrieved_evidence
):
    """
    Ask GPT-5.4-mini to judge whether one generated-summary claim
    is supported by retrieved evidence from the original record.
    """

    evidence_text = "\n\n".join(
        [
            (
                f"EVIDENCE RANK {item['rank']}\n"
                f"Date: {item['creation_timestamp']}\n"
                f"Section: {item['section_name']}\n"
                f"{item['text']}"
            )
            for item in retrieved_evidence
        ]
    )

    user_prompt = f"""
CLAIM:

{claim_text}


RETRIEVED ORIGINAL-RECORD EVIDENCE:

{evidence_text}
""".strip()

    response = client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": FAITHFULNESS_JUDGE_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )

    return json.loads(
        response.choices[0].message.content
    )

In [36]:
# ---------------------------------------------------------
# Step 12: Pilot Faithfulness support judgment
# ---------------------------------------------------------

test_indices = [
    0,
    len(test_claims) // 2,
    len(test_claims) - 1,
]

for idx in test_indices:

    claim = test_claims[idx]

    # Retrieve Top-10 evidence from the complete
    # original patient record.
    retrieved = retrieve_faithfulness_evidence(
        claim_text=claim["claim"],
        patient_evidence=patient_evidence,
        patient_embeddings=patient_embeddings,
    )

    # GPT-5.4-mini judges support.
    judgment = judge_faithfulness_claim(
        claim_text=claim["claim"],
        retrieved_evidence=retrieved,
    )

    print("\n" + "=" * 80)
    print(
        f"CLAIM {claim['claim_id']}: "
        f"{claim['claim']}"
    )
    print("LABEL:", judgment["label"])
    print("REASON:", judgment["reason"])
    print(
        "SUPPORTING EVIDENCE:",
        judgment["supporting_evidence_ranks"]
    )


CLAIM 1: On 21/12/2025, Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis.
LABEL: SUPPORTED
REASON: The record supports that on 21/12/2025 Hope Chinwo was seen/evaluated for severe left hip pain due to end-stage osteoarthritis, and the history notes it was gradual and worsening over the past 5 years.
SUPPORTING EVIDENCE: [2, 4, 6]

CLAIM 28: The surgical site was clean and dry.
LABEL: SUPPORTED
REASON: Multiple records state the surgical site/wound was clean and dry, with no bleeding or infection noted. This supports the claim that the surgical site was clean and dry.
SUPPORTING EVIDENCE: [1, 2, 5, 9]

CLAIM 55: She and her daughter received postoperative care instructions and hip-precaution instructions.
LABEL: SUPPORTED
REASON: The record shows discharge planning with the patient and next of kin/daughter, and that post-op care instructions were provided. It also explicitly states that hip precautions were emphasized and documente

In [37]:
# ---------------------------------------------------------
# Step 13: Prepare Faithfulness retrieval checkpoint
# ---------------------------------------------------------

FAITHFULNESS_RETRIEVAL_PATH = (
    EVALUATION_DIR
    / "faithfulness_bge_retrievals.json"
)

if FAITHFULNESS_RETRIEVAL_PATH.exists():

    with open(
        FAITHFULNESS_RETRIEVAL_PATH,
        "r"
    ) as f:
        faithfulness_retrievals = json.load(f)

else:
    faithfulness_retrievals = {}


print(
    "Workflow-patient retrievals already saved:",
    sum(
        len(patients)
        for patients in faithfulness_retrievals.values()
    )
)

Workflow-patient retrievals already saved: 0


In [38]:
# ---------------------------------------------------------
# Step 14: Production BGE retrieval for all Faithfulness claims
# ---------------------------------------------------------

for workflow, patients in faithfulness_atomic_claims.items():

    # Create workflow checkpoint if it does not exist yet.
    if workflow not in faithfulness_retrievals:
        faithfulness_retrievals[workflow] = {}

    print(f"\n{'=' * 80}")
    print("WORKFLOW:", workflow)
    print("=" * 80)

    for patient_idx, (person_id, claims) in enumerate(
        patients.items(),
        start=1
    ):

        person_id = str(person_id)

        # Skip workflow-patient combinations already saved.
        if person_id in faithfulness_retrievals[workflow]:
            print(
                f"Skipping patient {patient_idx}/{len(patients)} "
                "(already saved)"
            )
            continue

        print(
            f"Retrieving patient "
            f"{patient_idx}/{len(patients)} | "
            f"claims: {len(claims)}"
        )

        # -------------------------------------------------
        # Get this patient's COMPLETE original record chunks
        # -------------------------------------------------

        patient_mask = (
            section_chunks["person_id"].astype(str)
            == person_id
        )

        patient_evidence = (
            section_chunks[patient_mask]
            .copy()
            .reset_index(drop=True)
        )

        patient_embeddings = (
            section_chunk_embeddings[
                patient_mask.to_numpy()
            ]
        )

        if len(patient_evidence) == 0:
            raise ValueError(
                f"No source evidence found for {person_id}"
            )

        # -------------------------------------------------
        # Retrieve Top-10 evidence independently per claim
        # -------------------------------------------------

        patient_claim_retrievals = []

        for claim in claims:

            retrieved_evidence = (
                retrieve_faithfulness_evidence(
                    claim_text=claim["claim"],
                    patient_evidence=patient_evidence,
                    patient_embeddings=patient_embeddings,
                    top_k=FAITHFULNESS_TOP_K,
                )
            )

            patient_claim_retrievals.append({
                "claim_id": int(claim["claim_id"]),
                "claim": claim["claim"],
                "retrieved_evidence": retrieved_evidence,
            })

        # -------------------------------------------------
        # Save this workflow-patient result
        # -------------------------------------------------

        faithfulness_retrievals[
            workflow
        ][person_id] = patient_claim_retrievals

        # Checkpoint after every patient.
        with open(
            FAITHFULNESS_RETRIEVAL_PATH,
            "w"
        ) as f:
            json.dump(
                faithfulness_retrievals,
                f,
                indent=2
            )

        print("Saved.")


print("\nProduction retrieval complete.")


WORKFLOW: direct
Retrieving patient 1/50 | claims: 53
Saved.
Retrieving patient 2/50 | claims: 59
Saved.
Retrieving patient 3/50 | claims: 69
Saved.
Retrieving patient 4/50 | claims: 86
Saved.
Retrieving patient 5/50 | claims: 61
Saved.
Retrieving patient 6/50 | claims: 52
Saved.
Retrieving patient 7/50 | claims: 49
Saved.
Retrieving patient 8/50 | claims: 60
Saved.
Retrieving patient 9/50 | claims: 67
Saved.
Retrieving patient 10/50 | claims: 53
Saved.
Retrieving patient 11/50 | claims: 67
Saved.
Retrieving patient 12/50 | claims: 57
Saved.
Retrieving patient 13/50 | claims: 51
Saved.
Retrieving patient 14/50 | claims: 62
Saved.
Retrieving patient 15/50 | claims: 104
Saved.
Retrieving patient 16/50 | claims: 90
Saved.
Retrieving patient 17/50 | claims: 49
Saved.
Retrieving patient 18/50 | claims: 78
Saved.
Retrieving patient 19/50 | claims: 73
Saved.
Retrieving patient 20/50 | claims: 50
Saved.
Retrieving patient 21/50 | claims: 48
Saved.
Retrieving patient 22/50 | claims: 60
Saved.


In [39]:
# ---------------------------------------------------------
# Step 15: Validate Faithfulness BGE retrievals
# ---------------------------------------------------------

total_retrieved_claims = 0
invalid_retrievals = 0

for workflow in workflow_paths:

    patients = faithfulness_retrievals.get(
        workflow,
        {}
    )

    workflow_claims = sum(
        len(claims)
        for claims in patients.values()
    )

    total_retrieved_claims += workflow_claims

    print(
        f"{workflow}: "
        f"patients={len(patients)}, "
        f"claims={workflow_claims}, "
        f"IDs match="
        f"{set(patients.keys()) == study_patient_id_set}"
    )

    # Every claim should have exactly Top-10 evidence chunks.
    for person_id, claims in patients.items():
        for claim in claims:

            if len(
                claim["retrieved_evidence"]
            ) != FAITHFULNESS_TOP_K:
                invalid_retrievals += 1


print("\nTotal retrieved claims:", total_retrieved_claims)
print("Invalid retrievals:", invalid_retrievals)

direct: patients=50, claims=3000, IDs match=True
hierarchical: patients=50, claims=2905, IDs match=True
rag: patients=50, claims=2658, IDs match=True
rag_verification: patients=50, claims=2407, IDs match=True

Total retrieved claims: 10970
Invalid retrievals: 0


In [40]:
# ---------------------------------------------------------
# Check complete patient-record sizes
# ---------------------------------------------------------

record_sizes = []

for person_id, source_text in patient_source_documents.items():

    record_sizes.append({
        "person_id": person_id,
        "characters": len(source_text),
        "words": len(source_text.split()),
    })

record_sizes_df = pd.DataFrame(record_sizes)

print(
    record_sizes_df[
        ["characters", "words"]
    ].describe()
)

         characters        words
count     50.000000    50.000000
mean   19808.220000  2898.920000
std     6659.197182   958.822723
min     9034.000000  1380.000000
25%    13773.750000  2054.750000
50%    19886.500000  2914.500000
75%    24461.250000  3589.500000
max    37743.000000  5525.000000


In [41]:
# ---------------------------------------------------------
# Step 16: Patient-level Faithfulness evaluator
# ---------------------------------------------------------

FAITHFULNESS_PATIENT_SYSTEM_PROMPT = """
You are evaluating the faithfulness of generated clinical-summary claims
against a patient's complete original clinical record.

You will receive:
1. The patient's COMPLETE ORIGINAL CLINICAL RECORD.
2. Atomic claims extracted from four different generated summaries.

The original clinical record is the ONLY evidence source.

IMPORTANT RULES:

1. Evaluate every claim independently against the original clinical record.
2. Do NOT use claims from another workflow as evidence.
3. Do NOT use other claims from the same workflow as evidence.
4. Do NOT use outside medical knowledge.
5. Do NOT assume that a clinically plausible statement is true.
6. Reasonable paraphrases and synonymous clinical wording count as support.
7. Evaluate all clinically meaningful details, including diagnosis,
   medication, dose, timing, indication, anatomical site, procedure,
   investigation, progression, and outcome when present.
8. A claim is supported only by information present in the original record.
9. Return exactly one judgment for every supplied claim.

Assign exactly one label:

SUPPORTED:
The complete factual meaning of the claim is supported by the original record.

PARTIALLY_SUPPORTED:
The central claim is supported, but one or more clinically meaningful
details are unsupported, or only part of the factual claim is supported.

UNSUPPORTED:
The claim is not supported by the original record, or the original record
contradicts the claim.

Return ONLY valid JSON in this structure:

{
  "results": [
    {
      "workflow": "direct",
      "claim_id": 1,
      "label": "SUPPORTED",
      "reason": "Brief evidence-based explanation."
    }
  ]
}
""".strip()


def judge_patient_faithfulness(person_id):
    """
    Evaluate all four workflows for one patient in a single LLM call.
    """

    # Complete original patient record
    source_record = patient_source_documents[person_id]

    claim_blocks = []

    # Keep workflow identity explicit so claim IDs can repeat across workflows.
    for workflow in [
        "direct",
        "hierarchical",
        "rag",
        "rag_verification",
    ]:
        claims = faithfulness_atomic_claims[workflow][person_id]

        claim_text = "\n".join(
            f"- Claim ID {claim['claim_id']}: {claim['claim']}"
            for claim in claims
        )

        claim_blocks.append(
            f"""
WORKFLOW: {workflow}

{claim_text}
""".strip()
        )

    user_prompt = f"""
COMPLETE ORIGINAL CLINICAL RECORD:

{source_record}


============================================================
CLAIMS TO EVALUATE
============================================================

{chr(10).join(claim_blocks)}
""".strip()

    response = client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": FAITHFULNESS_PATIENT_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )

    return json.loads(response.choices[0].message.content)

In [42]:
# ---------------------------------------------------------
# Step 17: Pilot ONE patient-level Faithfulness call
# ---------------------------------------------------------

TEST_PERSON_ID = "05192757-942f-460d-b4ff-004ec39cc5ee"

pilot_result = judge_patient_faithfulness(TEST_PERSON_ID)

print("Returned results:", len(pilot_result["results"]))

# Show the first few judgments
print(
    json.dumps(
        pilot_result["results"][:10],
        indent=2
    )
)

Returned results: 254
[
  {
    "workflow": "direct",
    "claim_id": 1,
    "label": "SUPPORTED",
    "reason": "The record identifies Hope Chinwo as 75 years old and documents pre-op assessment for elective total left hip replacement on 21/12/25."
  },
  {
    "workflow": "direct",
    "claim_id": 2,
    "label": "SUPPORTED",
    "reason": "The record states the procedure was for end-stage OA of the left hip."
  },
  {
    "workflow": "direct",
    "claim_id": 3,
    "label": "SUPPORTED",
    "reason": "History documents gradual worsening over 5 years with stiffness, limited mobility, and worse pain with weight-bearing/activity."
  },
  {
    "workflow": "direct",
    "claim_id": 4,
    "label": "SUPPORTED",
    "reason": "Past medical history explicitly lists HTN."
  },
  {
    "workflow": "direct",
    "claim_id": 5,
    "label": "SUPPORTED",
    "reason": "Past medical history includes cataract surgery in 2018."
  },
  {
    "workflow": "direct",
    "claim_id": 6,
    "label": "S

In [43]:
# ---------------------------------------------------------
# Validate pilot output
# ---------------------------------------------------------

expected_pairs = set()

for workflow in [
    "direct",
    "hierarchical",
    "rag",
    "rag_verification",
]:
    for claim in faithfulness_atomic_claims[workflow][TEST_PERSON_ID]:
        expected_pairs.add(
            (workflow, int(claim["claim_id"]))
        )

returned_pairs = {
    (item["workflow"], int(item["claim_id"]))
    for item in pilot_result["results"]
}

valid_labels = {
    "SUPPORTED",
    "PARTIALLY_SUPPORTED",
    "UNSUPPORTED",
}

print("Expected claims:", len(expected_pairs))
print("Returned claims:", len(returned_pairs))
print("All claims returned:", returned_pairs == expected_pairs)

print(
    "All labels valid:",
    all(
        item["label"] in valid_labels
        for item in pilot_result["results"]
    )
)

print("Missing:", expected_pairs - returned_pairs)
print("Unexpected:", returned_pairs - expected_pairs)

Expected claims: 254
Returned claims: 254
All claims returned: True
All labels valid: True
Missing: set()
Unexpected: set()


In [44]:
# ---------------------------------------------------------
# Step 18: Run Faithfulness evaluation for all 50 patients
# One LLM call per patient, with checkpointing
# ---------------------------------------------------------

# Load existing results if restarting the notebook.
if FAITHFULNESS_RESULTS_PATH.exists():
    with open(FAITHFULNESS_RESULTS_PATH, "r") as f:
        faithfulness_results = json.load(f)
else:
    faithfulness_results = {}

# Save the already-completed pilot so we do not call it again.
if TEST_PERSON_ID not in faithfulness_results:
    faithfulness_results[TEST_PERSON_ID] = pilot_result

    with open(FAITHFULNESS_RESULTS_PATH, "w") as f:
        json.dump(faithfulness_results, f, indent=2)

print("Already completed:", len(faithfulness_results))


# Run remaining patients.
for i, person_id in enumerate(study_patient_ids, start=1):

    # Skip patients already completed.
    if person_id in faithfulness_results:
        print(f"[{i}/50] {person_id} — already complete")
        continue

    print(f"[{i}/50] Evaluating {person_id}...")

    result = judge_patient_faithfulness(person_id)

    # -----------------------------------------------------
    # Validate this patient's response BEFORE saving it.
    # -----------------------------------------------------

    expected_pairs = set()

    for workflow in [
        "direct",
        "hierarchical",
        "rag",
        "rag_verification",
    ]:
        for claim in faithfulness_atomic_claims[workflow][person_id]:
            expected_pairs.add(
                (workflow, int(claim["claim_id"]))
            )

    returned_pairs = {
        (item["workflow"], int(item["claim_id"]))
        for item in result["results"]
    }

    labels_are_valid = all(
        item["label"] in {
            "SUPPORTED",
            "PARTIALLY_SUPPORTED",
            "UNSUPPORTED",
        }
        for item in result["results"]
    )

    if returned_pairs != expected_pairs:
        raise ValueError(
            f"Claim mismatch for patient {person_id}. "
            f"Expected {len(expected_pairs)}, "
            f"received {len(returned_pairs)}."
        )

    if not labels_are_valid:
        raise ValueError(
            f"Invalid label returned for patient {person_id}."
        )

    # -----------------------------------------------------
    # Checkpoint immediately after each successful patient.
    # -----------------------------------------------------

    faithfulness_results[person_id] = result

    with open(FAITHFULNESS_RESULTS_PATH, "w") as f:
        json.dump(faithfulness_results, f, indent=2)

    print(
        f"[{i}/50] Complete — "
        f"{len(result['results'])} claims saved."
    )


print("\nFaithfulness evaluation finished.")
print("Patients completed:", len(faithfulness_results))
print("Saved to:", FAITHFULNESS_RESULTS_PATH)

Already completed: 1
[1/50] Evaluating 028998ee-babc-4096-9b28-001bc2f9a84e...
[1/50] Complete — 157 claims saved.
[2/50] Evaluating 04df53ea-55c1-48d9-84a1-1f15c133b29b...
[2/50] Complete — 192 claims saved.
[3/50] 05192757-942f-460d-b4ff-004ec39cc5ee — already complete
[4/50] Evaluating 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf...


ValueError: Claim mismatch for patient 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf. Expected 274, received 269.

In [45]:
# ---------------------------------------------------------
# Inspect claims missing from the failed patient response
# ---------------------------------------------------------

missing_pairs = expected_pairs - returned_pairs

print("Missing claims:", len(missing_pairs))
print(sorted(missing_pairs))

for workflow, claim_id in sorted(missing_pairs):
    claim = next(
        claim
        for claim in faithfulness_atomic_claims[workflow][person_id]
        if int(claim["claim_id"]) == claim_id
    )

    print(
        f"\n{workflow} | Claim {claim_id}\n"
        f"{claim['claim']}"
    )

Missing claims: 5
[('direct', 82), ('direct', 83), ('direct', 84), ('direct', 85), ('direct', 86)]

direct | Claim 82
The wound was healing without erythema or discharge.

direct | Claim 83
Vital signs remained stable.

direct | Claim 84
She was discharged with apixaban 2.5 mg twice daily for two weeks.

direct | Claim 85
She was discharged with a home exercise programme.

direct | Claim 86
She was discharged with follow-up with her GP in six weeks for wound review and outpatient physiotherapy referral.


In [46]:
# ---------------------------------------------------------
# Step 19: Recover missing Faithfulness judgments
# ---------------------------------------------------------

def judge_missing_faithfulness_claims(person_id, missing_pairs):
    """
    Evaluate only claims that were missing from a patient-level response.

    The patient's complete original clinical record remains the only
    evidence source.
    """

    source_record = patient_source_documents[person_id]

    claim_lines = []

    for workflow, claim_id in sorted(missing_pairs):

        claim = next(
            claim
            for claim in faithfulness_atomic_claims[workflow][person_id]
            if int(claim["claim_id"]) == int(claim_id)
        )

        claim_lines.append(
            f"- WORKFLOW: {workflow} | "
            f"CLAIM ID: {claim_id} | "
            f"CLAIM: {claim['claim']}"
        )

    user_prompt = f"""
COMPLETE ORIGINAL CLINICAL RECORD:

{source_record}


============================================================
CLAIMS TO EVALUATE
============================================================

{chr(10).join(claim_lines)}
""".strip()

    response = client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": FAITHFULNESS_PATIENT_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )

    return json.loads(response.choices[0].message.content)

In [47]:
# ---------------------------------------------------------
# Step 20: Recover and save the failed patient
# ---------------------------------------------------------

# Ask GPT only for the 5 missing judgments.
recovery_result = judge_missing_faithfulness_claims(
    person_id,
    missing_pairs
)

print("Recovered judgments:", len(recovery_result["results"]))
print(json.dumps(recovery_result["results"], indent=2))


# ---------------------------------------------------------
# Merge original 269 + recovered judgments
# ---------------------------------------------------------

merged_results = (
    result["results"]
    + recovery_result["results"]
)

merged_pairs = {
    (item["workflow"], int(item["claim_id"]))
    for item in merged_results
}

# Validate before saving.
print("\nExpected:", len(expected_pairs))
print("Returned after recovery:", len(merged_pairs))
print("Complete:", merged_pairs == expected_pairs)

assert merged_pairs == expected_pairs
assert len(merged_results) == len(expected_pairs)

assert all(
    item["label"] in {
        "SUPPORTED",
        "PARTIALLY_SUPPORTED",
        "UNSUPPORTED",
    }
    for item in merged_results
)


# ---------------------------------------------------------
# Save completed patient
# ---------------------------------------------------------

faithfulness_results[person_id] = {
    "results": merged_results
}

with open(FAITHFULNESS_RESULTS_PATH, "w") as f:
    json.dump(faithfulness_results, f, indent=2)

print("\nPatient successfully saved:", person_id)
print("Patients completed:", len(faithfulness_results))

Recovered judgments: 5
[
  {
    "workflow": "direct",
    "claim_id": 82,
    "label": "SUPPORTED",
    "reason": "The record states the surgical wound was clean, dry, and healing well with no erythema, swelling, or discharge."
  },
  {
    "workflow": "direct",
    "claim_id": 83,
    "label": "SUPPORTED",
    "reason": "Post-operative notes state cardiovascular and respiratory parameters were stable and vital signs were within normal limits."
  },
  {
    "workflow": "direct",
    "claim_id": 84,
    "label": "SUPPORTED",
    "reason": "The discharge plan explicitly says she was discharged with TTOs including apixaban 2.5 mg BD for 2 weeks."
  },
  {
    "workflow": "direct",
    "claim_id": 85,
    "label": "SUPPORTED",
    "reason": "The discharge plan includes continuing a home exercise programme."
  },
  {
    "workflow": "direct",
    "claim_id": 86,
    "label": "SUPPORTED",
    "reason": "The discharge plan states GP follow-up in 6 weeks for wound review and outpatient physio

In [48]:
# ---------------------------------------------------------
# Step 21: Continue Faithfulness evaluation
# with automatic missing-claim recovery
# ---------------------------------------------------------

VALID_LABELS = {
    "SUPPORTED",
    "PARTIALLY_SUPPORTED",
    "UNSUPPORTED",
}

for i, person_id in enumerate(study_patient_ids, start=1):

    # Skip patients already successfully saved.
    if person_id in faithfulness_results:
        print(f"[{i}/50] {person_id} — already complete")
        continue

    print(f"\n[{i}/50] Evaluating {person_id}...")

    # Main patient-level call.
    result = judge_patient_faithfulness(person_id)

    expected_pairs = set()

    for workflow in [
        "direct",
        "hierarchical",
        "rag",
        "rag_verification",
    ]:
        for claim in faithfulness_atomic_claims[workflow][person_id]:
            expected_pairs.add(
                (workflow, int(claim["claim_id"]))
            )

    # Store returned judgments by unique workflow + claim ID.
    collected = {
        (item["workflow"], int(item["claim_id"])): item
        for item in result["results"]
    }

    # -----------------------------------------------------
    # Recover any claims omitted by the main response.
    # -----------------------------------------------------

    missing_pairs = expected_pairs - set(collected)

    while missing_pairs:

        print(
            f"  Missing {len(missing_pairs)} claims — recovering..."
        )

        recovery = judge_missing_faithfulness_claims(
            person_id,
            missing_pairs
        )

        for item in recovery["results"]:
            key = (
                item["workflow"],
                int(item["claim_id"]),
            )

            # Only accept judgments we actually requested.
            if key in missing_pairs:
                collected[key] = item

        new_missing = expected_pairs - set(collected)

        # Prevent an endless loop if GPT returns none
        # of the requested missing claims.
        if new_missing == missing_pairs:
            raise ValueError(
                f"Recovery made no progress for {person_id}. "
                f"Still missing {len(new_missing)} claims."
            )

        missing_pairs = new_missing

    # -----------------------------------------------------
    # Final validation
    # -----------------------------------------------------

    final_results = list(collected.values())

    assert set(collected) == expected_pairs

    assert all(
        item["label"] in VALID_LABELS
        for item in final_results
    )

    print(
        f"  Complete: {len(final_results)} / "
        f"{len(expected_pairs)} claims"
    )

    # -----------------------------------------------------
    # Checkpoint completed patient
    # -----------------------------------------------------

    faithfulness_results[person_id] = {
        "results": final_results
    }

    with open(FAITHFULNESS_RESULTS_PATH, "w") as f:
        json.dump(faithfulness_results, f, indent=2)

    print(f"  Saved. Patients completed: {len(faithfulness_results)}")


print("\n--------------------------------")
print("Faithfulness LLM evaluation complete")
print("--------------------------------")
print("Patients:", len(faithfulness_results))
print("Saved to:", FAITHFULNESS_RESULTS_PATH)

[1/50] 028998ee-babc-4096-9b28-001bc2f9a84e — already complete
[2/50] 04df53ea-55c1-48d9-84a1-1f15c133b29b — already complete
[3/50] 05192757-942f-460d-b4ff-004ec39cc5ee — already complete
[4/50] 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf — already complete

[5/50] Evaluating 0f438665-d430-4adb-8acc-c3beed9e4942...
  Missing 11 claims — recovering...


AssertionError: 

In [49]:
# ---------------------------------------------------------
# Inspect invalid labels from the current patient
# ---------------------------------------------------------

invalid_items = [
    item
    for item in final_results
    if item.get("label") not in VALID_LABELS
]

print("Patient:", person_id)
print("Invalid labels:", len(invalid_items))

for item in invalid_items:
    print(
        "\nWorkflow:", item.get("workflow"),
        "\nClaim ID:", item.get("claim_id"),
        "\nLabel:", repr(item.get("label")),
        "\nReason:", item.get("reason")
    )

Patient: 0f438665-d430-4adb-8acc-c3beed9e4942
Invalid labels: 1

Workflow: hierarchical 
Claim ID: 7 
Label: 'He was refusing to drink fluids.' 
Reason: None


In [50]:
# ---------------------------------------------------------
# Inspect malformed claim
# ---------------------------------------------------------

bad_workflow = "hierarchical"
bad_claim_id = 7

bad_claim = next(
    claim
    for claim in faithfulness_atomic_claims[bad_workflow][person_id]
    if int(claim["claim_id"]) == bad_claim_id
)

print("Patient:", person_id)
print("Workflow:", bad_workflow)
print("Claim ID:", bad_claim_id)
print("Actual claim:", bad_claim["claim"])
print("\nMalformed result:")
print(json.dumps(invalid_items[0], indent=2))

Patient: 0f438665-d430-4adb-8acc-c3beed9e4942
Workflow: hierarchical
Claim ID: 7
Actual claim: At presentation, he was refusing fluids.

Malformed result:
{
  "workflow": "hierarchical",
  "claim_id": 7,
  "label": "He was refusing to drink fluids."
}


In [51]:
# ---------------------------------------------------------
# Repair the single malformed judgment
# ---------------------------------------------------------

repair_pair = {
    ("hierarchical", 7)
}

repair_result = judge_missing_faithfulness_claims(
    person_id,
    repair_pair
)

print(json.dumps(repair_result, indent=2))

{
  "results": [
    {
      "workflow": "hierarchical",
      "claim_id": 7,
      "label": "SUPPORTED",
      "reason": "The record states that parents reported reduced oral intake and that he had been refusing to drink fluids for 24 hours at presentation."
    }
  ]
}


In [52]:
# ---------------------------------------------------------
# Step 22: Replace malformed result and save current patient
# ---------------------------------------------------------

repair_item = repair_result["results"][0]
repair_key = (
    repair_item["workflow"],
    int(repair_item["claim_id"])
)

# Replace the malformed judgment with the repaired judgment.
collected[repair_key] = repair_item

final_results = list(collected.values())

# Final validation.
assert set(collected) == expected_pairs

assert all(
    item.get("label") in VALID_LABELS
    for item in final_results
)

assert len(final_results) == len(expected_pairs)

# Save completed patient.
faithfulness_results[person_id] = {
    "results": final_results
}

with open(FAITHFULNESS_RESULTS_PATH, "w") as f:
    json.dump(faithfulness_results, f, indent=2)

print("Patient successfully repaired and saved:", person_id)
print("Claims saved:", len(final_results))
print("Patients completed:", len(faithfulness_results))

Patient successfully repaired and saved: 0f438665-d430-4adb-8acc-c3beed9e4942
Claims saved: 289
Patients completed: 5


In [53]:
# ---------------------------------------------------------
# Step 23: Continue remaining Faithfulness evaluation
# Automatically recover missing OR malformed judgments
# ---------------------------------------------------------

VALID_LABELS = {
    "SUPPORTED",
    "PARTIALLY_SUPPORTED",
    "UNSUPPORTED",
}

WORKFLOWS = [
    "direct",
    "hierarchical",
    "rag",
    "rag_verification",
]


for i, person_id in enumerate(study_patient_ids, start=1):

    # Already validated and checkpointed.
    if person_id in faithfulness_results:
        print(f"[{i}/50] {person_id} — already complete")
        continue

    print(f"\n[{i}/50] Evaluating {person_id}...")

    # -----------------------------------------------------
    # Build the complete set of expected judgments.
    # -----------------------------------------------------

    expected_pairs = set()

    for workflow in WORKFLOWS:
        for claim in faithfulness_atomic_claims[workflow][person_id]:
            expected_pairs.add(
                (workflow, int(claim["claim_id"]))
            )

    # -----------------------------------------------------
    # Main patient-level LLM call.
    # -----------------------------------------------------

    result = judge_patient_faithfulness(person_id)

    collected = {}

    # Accept only expected results with valid labels.
    for item in result.get("results", []):

        try:
            key = (
                item["workflow"],
                int(item["claim_id"]),
            )
        except (KeyError, TypeError, ValueError):
            continue

        if (
            key in expected_pairs
            and item.get("label") in VALID_LABELS
        ):
            collected[key] = item

    # -----------------------------------------------------
    # Anything missing OR malformed is recovered.
    # -----------------------------------------------------

    recovery_pairs = expected_pairs - set(collected)

    while recovery_pairs:

        print(
            f"  Recovering {len(recovery_pairs)} "
            f"missing/malformed judgments..."
        )

        recovery = judge_missing_faithfulness_claims(
            person_id,
            recovery_pairs,
        )

        previous_count = len(collected)

        for item in recovery.get("results", []):

            try:
                key = (
                    item["workflow"],
                    int(item["claim_id"]),
                )
            except (KeyError, TypeError, ValueError):
                continue

            if (
                key in recovery_pairs
                and item.get("label") in VALID_LABELS
            ):
                collected[key] = item

        recovery_pairs = expected_pairs - set(collected)

        # Prevent an infinite retry loop.
        if (
            len(collected) == previous_count
            and recovery_pairs
        ):
            raise ValueError(
                f"Recovery made no progress for {person_id}. "
                f"Still need {len(recovery_pairs)} judgments."
            )

    # -----------------------------------------------------
    # Final deterministic validation
    # -----------------------------------------------------

    assert set(collected) == expected_pairs
    assert len(collected) == len(expected_pairs)

    assert all(
        item["label"] in VALID_LABELS
        for item in collected.values()
    )

    # Sort results for cleaner/reproducible JSON.
    final_results = sorted(
        collected.values(),
        key=lambda x: (
            WORKFLOWS.index(x["workflow"]),
            int(x["claim_id"]),
        ),
    )

    # -----------------------------------------------------
    # Checkpoint only after patient is fully valid.
    # -----------------------------------------------------

    faithfulness_results[person_id] = {
        "results": final_results
    }

    with open(FAITHFULNESS_RESULTS_PATH, "w") as f:
        json.dump(faithfulness_results, f, indent=2)

    print(
        f"  Complete: {len(final_results)} / "
        f"{len(expected_pairs)} claims"
    )
    print(
        f"  Saved. Patients completed: "
        f"{len(faithfulness_results)}/50"
    )


print("\n--------------------------------")
print("Faithfulness LLM evaluation complete")
print("--------------------------------")
print("Patients:", len(faithfulness_results))
print("Saved to:", FAITHFULNESS_RESULTS_PATH)

[1/50] 028998ee-babc-4096-9b28-001bc2f9a84e — already complete
[2/50] 04df53ea-55c1-48d9-84a1-1f15c133b29b — already complete
[3/50] 05192757-942f-460d-b4ff-004ec39cc5ee — already complete
[4/50] 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf — already complete
[5/50] 0f438665-d430-4adb-8acc-c3beed9e4942 — already complete

[6/50] Evaluating 136c7916-4f9b-4e5c-bf01-77e9d2c681a2...
  Recovering 1 missing/malformed judgments...
  Complete: 201 / 201 claims
  Saved. Patients completed: 6/50

[7/50] Evaluating 137b8481-4f1d-4b7f-babd-20f7117023ad...
  Recovering 2 missing/malformed judgments...
  Complete: 172 / 172 claims
  Saved. Patients completed: 7/50

[8/50] Evaluating 1705dd0f-011a-492c-b006-b27e03f2f4ed...
  Recovering 4 missing/malformed judgments...
  Complete: 252 / 252 claims
  Saved. Patients completed: 8/50

[9/50] Evaluating 1dbe23dc-0d1e-431b-81eb-497282b46a14...
  Recovering 59 missing/malformed judgments...
  Complete: 304 / 304 claims
  Saved. Patients completed: 9/50

[10/50] Eva

In [54]:
# ---------------------------------------------------------
# Step 24: Final Faithfulness integrity validation
# ---------------------------------------------------------

# Reload from disk so we validate the actual saved artifact.
with open(FAITHFULNESS_RESULTS_PATH, "r") as f:
    faithfulness_results = json.load(f)

VALID_LABELS = {
    "SUPPORTED",
    "PARTIALLY_SUPPORTED",
    "UNSUPPORTED",
}

WORKFLOWS = [
    "direct",
    "hierarchical",
    "rag",
    "rag_verification",
]

total_expected = 0
total_returned = 0

missing = []
unexpected = []
invalid_labels = []
duplicate_pairs = []

for person_id in study_patient_ids:

    # Expected workflow + claim ID combinations
    expected_pairs = set()

    for workflow in WORKFLOWS:
        for claim in faithfulness_atomic_claims[workflow][person_id]:
            expected_pairs.add(
                (workflow, int(claim["claim_id"]))
            )

    total_expected += len(expected_pairs)

    # Saved judgments
    patient_results = faithfulness_results[person_id]["results"]

    seen = set()

    for item in patient_results:

        key = (
            item["workflow"],
            int(item["claim_id"]),
        )

        if key in seen:
            duplicate_pairs.append((person_id, key))

        seen.add(key)

        if item.get("label") not in VALID_LABELS:
            invalid_labels.append(
                (
                    person_id,
                    key,
                    item.get("label"),
                )
            )

    total_returned += len(seen)

    missing.extend(
        (person_id, key)
        for key in expected_pairs - seen
    )

    unexpected.extend(
        (person_id, key)
        for key in seen - expected_pairs
    )


print("Patients:", len(faithfulness_results))
print("Expected judgments:", total_expected)
print("Returned judgments:", total_returned)

print("Missing:", len(missing))
print("Unexpected:", len(unexpected))
print("Duplicates:", len(duplicate_pairs))
print("Invalid labels:", len(invalid_labels))

Patients: 50
Expected judgments: 10970
Returned judgments: 10970
Missing: 0
Unexpected: 0
Duplicates: 0
Invalid labels: 0


In [55]:
# ---------------------------------------------------------
# Step 25: Calculate patient-level Faithfulness scores
# ---------------------------------------------------------

LABEL_POINTS = {
    "SUPPORTED": 2,
    "PARTIALLY_SUPPORTED": 1,
    "UNSUPPORTED": 0,
}

faithfulness_score_rows = []

for person_id in study_patient_ids:

    patient_results = faithfulness_results[person_id]["results"]

    for workflow in WORKFLOWS:

        # Judgments belonging to this workflow
        workflow_results = [
            item
            for item in patient_results
            if item["workflow"] == workflow
        ]

        n_claims = len(workflow_results)

        supported = sum(
            item["label"] == "SUPPORTED"
            for item in workflow_results
        )

        partially_supported = sum(
            item["label"] == "PARTIALLY_SUPPORTED"
            for item in workflow_results
        )

        unsupported = sum(
            item["label"] == "UNSUPPORTED"
            for item in workflow_results
        )

        # Weighted Faithfulness score:
        # SUPPORTED = 2, PARTIAL = 1, UNSUPPORTED = 0
        score = (
            sum(
                LABEL_POINTS[item["label"]]
                for item in workflow_results
            )
            / (2 * n_claims)
        ) * 100

        faithfulness_score_rows.append({
            "person_id": person_id,
            "workflow": workflow,
            "claim_count": n_claims,
            "supported": supported,
            "partially_supported": partially_supported,
            "unsupported": unsupported,
            "faithfulness_score": score,
        })


faithfulness_scores_df = pd.DataFrame(
    faithfulness_score_rows
)

print("Rows:", len(faithfulness_scores_df))

print(
    faithfulness_scores_df
    .groupby("workflow")["faithfulness_score"]
    .agg(["mean", "median", "std", "min", "max"])
    .round(2)
)

Rows: 200
                   mean  median   std    min    max
workflow                                           
direct            99.36   100.0  1.10  94.85  100.0
hierarchical      99.53   100.0  0.79  97.06  100.0
rag               98.18   100.0  3.80  84.44  100.0
rag_verification  98.54   100.0  2.81  88.00  100.0


In [56]:
# ---------------------------------------------------------
# Step 26: Faithfulness raw label counts and proportions
# ---------------------------------------------------------

label_summary = (
    faithfulness_scores_df
    .groupby("workflow")
    .agg(
        total_claims=("claim_count", "sum"),
        supported=("supported", "sum"),
        partially_supported=("partially_supported", "sum"),
        unsupported=("unsupported", "sum"),
    )
)

# Calculate proportions of all generated claims.
label_summary["supported_pct"] = (
    label_summary["supported"]
    / label_summary["total_claims"]
    * 100
)

label_summary["partially_supported_pct"] = (
    label_summary["partially_supported"]
    / label_summary["total_claims"]
    * 100
)

label_summary["unsupported_pct"] = (
    label_summary["unsupported"]
    / label_summary["total_claims"]
    * 100
)

print(
    label_summary.round(2)
)

                  total_claims  supported  partially_supported  unsupported  \
workflow                                                                      
direct                    3000       2969                   23            8   
hierarchical              2905       2879                   19            7   
rag                       2658       2603                   21           34   
rag_verification          2407       2366                   14           27   

                  supported_pct  partially_supported_pct  unsupported_pct  
workflow                                                                   
direct                    98.97                     0.77             0.27  
hierarchical              99.10                     0.65             0.24  
rag                       97.93                     0.79             1.28  
rag_verification          98.30                     0.58             1.12  


In [57]:
# ---------------------------------------------------------
# Step 27: Check paired Faithfulness differences
# ---------------------------------------------------------

from scipy.stats import shapiro
from itertools import combinations

# Patient × workflow matrix
faithfulness_wide = (
    faithfulness_scores_df
    .pivot(
        index="person_id",
        columns="workflow",
        values="faithfulness_score",
    )
    [WORKFLOWS]
)

print("Shape:", faithfulness_wide.shape)
print("Missing values:", faithfulness_wide.isna().sum().sum())

print("\nShapiro-Wilk tests on paired differences:")

for workflow_a, workflow_b in combinations(WORKFLOWS, 2):

    differences = (
        faithfulness_wide[workflow_a]
        - faithfulness_wide[workflow_b]
    )

    stat, p = shapiro(differences)

    print(
        f"{workflow_a} vs {workflow_b}: "
        f"W={stat:.4f}, p={p:.6g}"
    )

Shape: (50, 4)
Missing values: 0

Shapiro-Wilk tests on paired differences:
direct vs hierarchical: W=0.9096, p=0.00101459
direct vs rag: W=0.6601, p=1.69762e-09
direct vs rag_verification: W=0.7891, p=4.95203e-07
hierarchical vs rag: W=0.6674, p=2.24787e-09
hierarchical vs rag_verification: W=0.7377, p=4.19798e-08
rag vs rag_verification: W=0.5934, p=1.55586e-10


In [58]:
# ---------------------------------------------------------
# Step 28: Friedman test for Faithfulness
# ---------------------------------------------------------

from scipy.stats import friedmanchisquare

friedman_stat, friedman_p = friedmanchisquare(
    faithfulness_wide["direct"],
    faithfulness_wide["hierarchical"],
    faithfulness_wide["rag"],
    faithfulness_wide["rag_verification"],
)

print(f"Friedman chi-square: {friedman_stat:.6f}")
print(f"p-value: {friedman_p:.12g}")

Friedman chi-square: 1.515152
p-value: 0.678777809624


In [59]:
# ---------------------------------------------------------
# Step 29: Save final Faithfulness evaluation outputs
# ---------------------------------------------------------

# 1. Patient-level scores
faithfulness_patient_scores_path = (
    EVALUATION_DIR / "faithfulness_patient_scores.csv"
)

faithfulness_scores_df.to_csv(
    faithfulness_patient_scores_path,
    index=False
)


# 2. Workflow-level descriptive statistics
faithfulness_workflow_summary = (
    faithfulness_scores_df
    .groupby("workflow")["faithfulness_score"]
    .agg(["mean", "median", "std", "min", "max"])
    .reset_index()
)

faithfulness_workflow_summary_path = (
    EVALUATION_DIR / "faithfulness_workflow_summary.csv"
)

faithfulness_workflow_summary.to_csv(
    faithfulness_workflow_summary_path,
    index=False
)


# 3. Raw label counts and proportions
faithfulness_label_summary_path = (
    EVALUATION_DIR / "faithfulness_label_summary.csv"
)

label_summary.reset_index().to_csv(
    faithfulness_label_summary_path,
    index=False
)


# ---------------------------------------------------------
# Confirm saved files
# ---------------------------------------------------------

print("Saved:")
print(faithfulness_patient_scores_path)
print(faithfulness_workflow_summary_path)
print(faithfulness_label_summary_path)

print("\nWorkflow summary:")
print(
    faithfulness_workflow_summary.round(2).to_string(index=False)
)

print("\nLabel summary:")
print(
    label_summary.round(2).to_string()
)

Saved:
/Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation/faithfulness_patient_scores.csv
/Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation/faithfulness_workflow_summary.csv
/Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation/faithfulness_label_summary.csv

Workflow summary:
        workflow  mean  median  std   min   max
          direct 99.36   100.0 1.10 94.85 100.0
    hierarchical 99.53   100.0 0.79 97.06 100.0
             rag 98.18   100.0 3.80 84.44 100.0
rag_verification 98.54   100.0 2.81 88.00 100.0

Label summary:
                  total_claims  supported  partially_supported  unsupported  supported_pct  partially_supported_pct  unsupported_pct
workflow                                                                                                                            
direct                    3000       2969                   23            8          98.